# Scenario Playground Step 16: Confidence Diagnostics

Objective: combine accepted Step 15 sensitivity evidence with historical ranges, joint feature familiarity, walk-forward errors, VIX-regime errors, residual extremes, and state freshness for all 12 Ridge targets.

Confidence is rule based and non-compensatory. No diagnostic can average away weak Step 13 evidence, and unavailable all-target XGBoost agreement is excluded without penalty or bonus.

In [ ]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.confidence_diagnostics import (
    XGBOOST_AGREEMENT_STATUS,
    build_confidence_artifacts,
    persist_confidence_artifacts,
)
from scenario_modeling.src.scenario_explanations import build_explanation_artifacts
from scenario_modeling.src.scenario_inference import build_research_inference_engine

DIAGNOSTIC_AS_OF_DATE = date.today().isoformat()
engine = build_research_inference_engine()
explanations = build_explanation_artifacts(engine=engine)
artifacts = build_confidence_artifacts(
    diagnostic_as_of_date=DIAGNOSTIC_AS_OF_DATE,
    engine=engine,
    explanation_artifacts=explanations,
)
output_paths = persist_confidence_artifacts(artifacts)
summary = artifacts["summary"]
{
    "python_executable": sys.executable,
    "completion_gate": summary["completion_gate"],
    "diagnostic_as_of_date": DIAGNOSTIC_AS_OF_DATE,
    "confidence_counts": summary["overall_confidence_counts"],
    "locked_final_test": summary["locked_final_test"],
}


## Historical ranges and percentiles

All eight scenario shocks and all 45 current-state features are compared with the same 464 authorized pre-final-test rows. Values are reported unchanged; no clipping or winsorization occurs.

In [ ]:
inputs = artifacts["input_diagnostics"]
display(inputs.groupby(["input_role", "warning_status"]).size().rename("count").to_frame())
display(inputs.loc[
    ~inputs["warning_status"].eq("WITHIN_TRAINING_RANGE"),
    ["input_role", "contract_input", "feature_name", "value", "historical_minimum", "historical_maximum", "empirical_percentile", "warning_status"],
])
assert len(inputs) == 53
assert inputs["input_role"].eq("scenario_shock").sum() == 8
assert inputs["input_role"].eq("current_state").sum() == 45


## Familiarity of the complete 58-feature combination

Nearest-neighbor distance is measured after the fitted training median imputer and scaler. The scenario distance is compared with leave-one-out nearest-neighbor distances among the 464 historical rows.

In [ ]:
distance = artifacts["joint_feature_distance"]
display(pd.DataFrame(distance["rows"]).T)
display(pd.Series(distance["reference_distribution"], name="value").to_frame())
assert distance["reference_distribution"]["feature_count"] == 58
assert distance["reference_distribution"]["training_rows"] == 464


## Per-target research confidence

The output preserves each target's Step 13 gate, then adds scenario-specific warnings. The nine weak-evidence targets cannot be upgraded by range or regime diagnostics.

In [ ]:
targets = artifacts["target_confidence_diagnostics"]
display(targets[[
    "target", "unit", "step13_evaluation_classification",
    "overall_research_confidence", "walk_forward_mae",
    "walk_forward_oos_r_squared", "current_vix_regime",
    "similar_regime_mae", "negligible_scenario_input_count",
    "unstable_scenario_input_count", "state_freshness_warning_count",
]])
assert len(targets) == 12
assert not targets["overall_research_confidence"].str.contains("HIGH").any()


## Structured warnings and state freshness

Input range, state extrapolation, model performance, scenario responsiveness, regime, and state freshness remain distinct categories.

In [ ]:
warnings = artifacts["warning_records"]
display(warnings.groupby(["category", "severity"]).size().rename("warning_count").to_frame())
display(pd.Series(artifacts["state_freshness"], name="value").to_frame())
assert set(warnings["category"]) <= {
    "input_range", "state_extrapolation", "model_performance",
    "scenario_responsiveness", "regime", "state_freshness",
}


## Intentionally postponed XGBoost agreement

All-target XGBoost agreement is unavailable. It receives no score, no penalty, and no bonus; it can be added only in a later version after proper all-target evaluation.

In [ ]:
display(targets[[
    "target", "xgboost_agreement_status",
    "xgboost_agreement_included", "xgboost_agreement_score",
]])
assert targets["xgboost_agreement_status"].eq(XGBOOST_AGREEMENT_STATUS).all()
assert not targets["xgboost_agreement_included"].any()
assert targets["xgboost_agreement_score"].isna().all()


## Acceptance checks and Step 17 handoff

The frozen explanation and confidence schemas are suitable for inclusion beside the 12 fitted Ridge pipelines in a later reloadable artifact bundle. This notebook does not export those pipelines.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
display(pd.Series(summary["step17_readiness"], name="ready").to_frame())
assert checks.all()
assert summary["completion_gate"] == "STEP_16_CONFIDENCE_DIAGNOSTICS_COMPLETE"
assert summary["locked_final_test"]["target_rows_parsed"] == 0
assert summary["locked_final_test"]["access_count"] == 0
assert summary["step17_readiness"]["ready_for_ridge_artifact_export_after_acceptance"]
print(summary["completion_gate"])


## Decision

Step 16 is accepted. Extreme or unfamiliar scenarios are flagged, weak evidence cannot be averaged into strength, XGBoost unavailability is neutral, and the project may proceed to the separate Step 17 Ridge artifact-export boundary.